# H&M Recommender — EDA

Quick look at interaction counts, popularity skew, and cold-start prevalence in the H&M dataset. The goal here is just to sanity-check assumptions that feed into the two-tower pipeline (`src/data/sample.py`, `src/data/preprocess.py`) — e.g. how skewed is item popularity (informs whether in-batch negatives are a reasonable choice vs. popularity-corrected sampling), and how common is cold start (informs whether content features in the item tower are pulling real weight).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from src.config import RAW_DIR

articles = pd.read_csv(RAW_DIR / "articles.csv", dtype={"article_id": str})
customers = pd.read_csv(RAW_DIR / "customers.csv")
transactions = pd.read_csv(
    RAW_DIR / "transactions_train.csv",
    dtype={"article_id": str, "customer_id": "string"},
    parse_dates=["t_dat"],
)
print(f"transactions: {len(transactions):,}  customers: {len(customers):,}  articles: {len(articles):,}")

## Weekly transaction volume
Confirms the date range and whether volume is stable enough that a trailing N-week window (used by `src/data/sample.py`) is a representative dev sample.

In [ ]:
weekly = transactions.set_index("t_dat").resample("W").size()
weekly.plot(figsize=(10, 3), title="Transactions per week")
plt.show()
print(f"date range: {transactions['t_dat'].min().date()} to {transactions['t_dat'].max().date()}")

## Interaction counts per customer
Most users have very few purchases — this is why we use in-batch negatives and rely on content (categorical) features rather than purely collaborative customer_id embeddings to generalize.

In [ ]:
per_customer = transactions.groupby("customer_id").size()
print(per_customer.describe())
per_customer.clip(upper=per_customer.quantile(0.99)).hist(bins=50, figsize=(8, 3))
plt.title("Transactions per customer (clipped at 99th pct)")
plt.show()

## Item popularity skew
What fraction of all transactions the top-K most popular articles account for — a highly skewed catalog means a naive popularity baseline is a strong competitor and the model needs to do better than just memorizing bestsellers.

In [ ]:
per_article = transactions.groupby("article_id").size().sort_values(ascending=False)
total = per_article.sum()
for k in (12, 100, 1000):
    share = per_article.head(k).sum() / total
    print(f"top {k:>5} articles account for {share:.1%} of all transactions")

## Cold-start prevalence
Customers/articles with zero purchases never appear in the transactions log, so the model only ever sees customers/articles with at least one interaction during training — this matters for how much of the catalog/customer base the trained embeddings actually cover.

In [ ]:
customers_with_txn = transactions["customer_id"].nunique()
articles_with_txn = transactions["article_id"].nunique()
print(f"customers with >=1 transaction: {customers_with_txn:,} / {len(customers):,} "
      f"({customers_with_txn / len(customers):.1%})")
print(f"articles with >=1 transaction: {articles_with_txn:,} / {len(articles):,} "
      f"({articles_with_txn / len(articles):.1%})")